<a href="https://colab.research.google.com/github/cheriet-noureddine/product_sales/blob/main/ProjetPythonAvancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets matplotlib
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# 1. Install required libraries
!pip install transformers datasets matplotlib -q

In [ ]:
!pip uninstall -y transformers
!pip install -U transformers huggingface_hub -q


In [ ]:
!pip install -U huggingface_hub transformers -q


In [ ]:
# 1. Install required libraries
!pip install transformers datasets matplotlib -q

# 2. Define system prompt
system_prompt = """You are a compassionate intake assistant. You do not give medical advice.
You support users by asking clarifying questions and helping them feel heard."""

# 3. Build example dataset
data = [
    {"input": "I feel anxious lately.", "target": "Can you tell me more about when this anxiety started?"},
    {"input": "I can't sleep well.", "target": "How long have you been experiencing sleep issues?"},
    {"input": "I'm overwhelmed with work.", "target": "What part of your work feels most overwhelming right now?"}
]

# 4. Tokenize inputs/labels
from datasets import Dataset
from transformers import AutoTokenizer

dataset = Dataset.from_list(data)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

def preprocess(example):
    input_text = system_prompt + " " + example["input"]
    model_inputs = tokenizer(input_text, truncation=True, padding="max_length", max_length=64)
    labels = tokenizer(example["target"], truncation=True, padding="max_length", max_length=64)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.3)

# 5. Load FLAN-T5 model
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# 6. Set up Seq2Seq training arguments
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    predict_with_generate=True,
    logging_steps=10,
    save_strategy="no"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer
)

# 7. Train model
train_output = trainer.train()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
import matplotlib.pyplot as plt

# Extract loss values from Trainer logs
train_loss = [log["loss"] for log in trainer.state.log_history if "loss" in log]
eval_loss = [log["eval_loss"] for log in trainer.state.log_history if "eval_loss" in log]
steps = [log["step"] for log in trainer.state.log_history if "loss" in log]

# Plot training and evaluation loss
plt.figure(figsize=(8,5))
plt.plot(steps, train_loss, label="Train Loss")
plt.plot([log["step"] for log in trainer.state.log_history if "eval_loss" in log],
         eval_loss, label="Eval Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Learning Curves")
plt.legend()
plt.show()


In [ ]:
import gradio as gr


def greet(name):
    return "Hello " + name


demo = gr.Interface(fn=greet, inputs="text", outputs="text")

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f61e5b9d835ac2f279.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
